In [28]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D
from scipy import stats

from esda.moran import Moran
from libpysal.weights import KNN
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import xskillscore as xs
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
import re

from collections import defaultdict
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [29]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 4
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 16, number of workers: 4, processes: False


In [30]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jwille//proxy/44293/status,
Dashboard: /user/jwille//proxy/44293/status,Workers: 4
Total threads: 64,Total memory: 3.64 TiB
Status: running,Using processes: False
Comm: inproc://129.132.32.66/1383988/52,Workers: 4
Dashboard: /user/jwille//proxy/44293/status,Total threads: 64
Started: Just now,Total memory: 3.64 TiB
Comm: inproc://129.132.32.66/1383988/55,Total threads: 16
Dashboard: /user/jwille//proxy/34567/status,Memory: 0.91 TiB
Nanny: None,


In [31]:
def temporal_auto(data):
    # 4. Create a lagged version (shift by 1 day)
    data_lag1 = data.shift(time=1)
    # 5. Compute lag-1 autocorrelation
    #    Definition: corr(binary(t), binary(t−1)) along time dimension
    r_pear = xr.corr(data, data_lag1, dim='time')

    return r_pear
    

In [32]:
latitude = [30,60]
months = [5,6,7,8,9,10]
dummy = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/1H/dummy.nc')['PRECT']
land_sea = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/24H/mask.nc')['MASK']
land_sea = land_sea[0,:,:]
indexer = ((land_sea.lat>=latitude[0]) & (land_sea.lat<=latitude[1]))
land_sea = land_sea.where(indexer,drop=True)
indexer = ((dummy.lat>=latitude[0]) & (dummy.lat<=latitude[1]))

dummy = dummy.where(indexer,drop=True)
land_sea = land_sea.assign_coords(lat=dummy.lat, lon=dummy.lon)
lat_weights = xr.DataArray(
    np.cos(np.deg2rad(land_sea['lat'])),
    dims=['lat'],
    coords={'lat': land_sea['lat']}
)

## IFS deep on

In [14]:

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_on/daily_precip/cesm2_grid')
files = sorted([str(f) for f in data_path.glob('daysum*')])[:]
ifs_on = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']

ifs_on = ifs_on.where(ifs_on.time.dt.month.isin(months), drop=True)
indexer = ((ifs_on.lat>=latitude[0]) & (ifs_on.lat<=latitude[1]))

ifs_on = ifs_on.where(indexer,drop=True)
ifs_on = ifs_on.load()

ifs_on = ifs_on.where(land_sea > 0, np.nan)

temporal_temp_pear = temporal_auto(ifs_on)
temporal_temp_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_2.8km_on_cesm2_grid_30-60N_temporal_corr_nonbinary.nc')


## IFS deep off

In [7]:

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/daily_precip/cesm2_grid')
files = sorted([str(f) for f in data_path.glob('daysum*')])[:]
ifs_off = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']

ifs_off = ifs_off.where(ifs_off.time.dt.month.isin(months), drop=True)
indexer = ((ifs_off.lat>=latitude[0]) & (ifs_off.lat<=latitude[1]))

ifs_off = ifs_off.where(indexer,drop=True)
ifs_off = ifs_off.load()

ifs_off = ifs_off.where(land_sea > 0, np.nan)

temporal_temp_pear = temporal_auto(ifs_off)
temporal_temp_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_2.8km_off_cesm2_grid_30-60N_temporal_corr_nonbinary.nc')


NameError: name 'model' is not defined

## IFS cycle3

In [34]:

years = range(2020, 2025)
results_temporal_pear = []
for year in years:
    print(f"Processing year {year}...")

    data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/daily_precip/cesm2_grid')
    files = sorted([str(f) for f in data_path.glob('daysum_cesm2_interp_*_??_'+str(year)+'.nc')])[:]
    ifs_cycle_3 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']

    ifs_cycle_3 = ifs_cycle_3.where(ifs_cycle_3.time.dt.month.isin(months), drop=True)
    indexer = ((ifs_cycle_3.lat>=latitude[0]) & (ifs_cycle_3.lat<=latitude[1]))

    ifs_cycle_3 = ifs_cycle_3.where(indexer,drop=True)
    ifs_cycle_3 = ifs_cycle_3.load()

    ifs_cycle_3 = ifs_cycle_3.where(land_sea > 0, np.nan)

    temporal_temp_pear = temporal_auto(ifs_cycle_3)
    results_temporal_pear.append(temporal_temp_pear)
    
ifs_cycle3_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")

ifs_cycle3_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_cycle_3_cesm2_grid_30-60N_temporal_corr_nonbinary.nc')


Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...


## IFS cycle4

In [36]:

years = range(1990, 2020)
results_temporal_pear = []
for year in years:
    print(f"Processing year {year}...")

    data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/1990-2020/daily_precip/cesm2_grid')
    files = sorted([str(f) for f in data_path.glob('daysum_cesm2_interp_*_??_'+str(year)+'.nc')])[:]
    ifs_cycle_4 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']

    ifs_cycle_4 = ifs_cycle_4.where(ifs_cycle_4.time.dt.month.isin(months), drop=True)
    indexer = ((ifs_cycle_4.lat>=latitude[0]) & (ifs_cycle_4.lat<=latitude[1]))

    ifs_cycle_4 = ifs_cycle_4.where(indexer,drop=True)
    ifs_cycle_4 = ifs_cycle_4.load()

    ifs_cycle_4 = ifs_cycle_4.where(land_sea > 0, np.nan)

    temporal_temp_pear = temporal_auto(ifs_cycle_4)
    results_temporal_pear.append(temporal_temp_pear)
    
ifs_cycle_4_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")

ifs_cycle_4_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_cycle_4_cesm2_grid_30-60N_temporal_corr_nonbinary.nc')


Processing year 1990...
Processing year 1991...
Processing year 1992...
Processing year 1993...
Processing year 1994...
Processing year 1995...
Processing year 1996...
Processing year 1997...
Processing year 1998...
Processing year 1999...
Processing year 2000...
Processing year 2001...
Processing year 2002...
Processing year 2003...
Processing year 2004...
Processing year 2005...
Processing year 2006...
Processing year 2007...
Processing year 2008...
Processing year 2009...
Processing year 2010...
Processing year 2011...
Processing year 2012...
Processing year 2013...
Processing year 2014...
Processing year 2015...
Processing year 2016...
Processing year 2017...
Processing year 2018...
Processing year 2019...


In [37]:
print(ifs_cycle3_temporal_corr_pear)

<xarray.DataArray 'tp' (year: 30, lat: 32, lon: 288)> Size: 2MB
array([[[ 0.46308776,  0.30813614,  0.12546633, ...,  0.13668307,
          0.30549992,  0.47674276],
        [ 0.28585587,  0.37341929,  0.03285473, ...,  0.09287931,
          0.36601972,  0.39237597],
        [ 0.26224539,  0.29738171,  0.15567973, ...,  0.03900328,
          0.34626786,  0.22852157],
        ...,
        [        nan,         nan,         nan, ...,  0.32124415,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan]],

       [[ 0.11479647,  0.02644336,  0.02333085, ...,  0.45718179,
          0.50396448,  0.40866872],
        [ 0.0300231 ,  0.03571324, -0.0137835 , ...,  0.27533404,
          0.11849984,  0.05372547],
        [ 0.14593245,  0.23281739,  0.03884405, ...,  0.28634941,
          0.04632746,  0.0585564 ],
...

## ICON cycle3

In [39]:

years = range(2020, 2025)
results_temporal_pear = []
for year in years:
    print(f"Processing year {year}...")

    data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/daily_precip/cesm2_grid')
    files = sorted([str(f) for f in data_path.glob('daysum_cesm2_interp_*_??_'+str(year)+'.nc')])[:]
    icon_cycle_3 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['pr']

    icon_cycle_3 = icon_cycle_3.where(icon_cycle_3.time.dt.month.isin(months), drop=True)
    indexer = ((icon_cycle_3.lat>=latitude[0]) & (icon_cycle_3.lat<=latitude[1]))

    icon_cycle_3 = icon_cycle_3.where(indexer,drop=True)
    icon_cycle_3 = icon_cycle_3.load()

    icon_cycle_3 = icon_cycle_3.where(land_sea > 0, np.nan)

    temporal_temp_pear = temporal_auto(icon_cycle_3)
    results_temporal_pear.append(temporal_temp_pear)
    
icon_cycle3_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")

icon_cycle3_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/icon_cycle_3_cesm2_grid_30-60N_temporal_corr_nonbinary.nc')


Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...


## ICON cycle4

In [40]:

years = range(2020, 2050)
results_temporal_pear = []
for year in years:
    print(f"Processing year {year}...")

    data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/2020-2050/daily_precip/cesm2_grid')
    files = sorted([str(f) for f in data_path.glob('daysum_cesm2_interp_*_??_'+str(year)+'.nc')])[:]
    icon_cycle_4 = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['pr']

    icon_cycle_4 = icon_cycle_4.where(icon_cycle_4.time.dt.month.isin(months), drop=True)
    indexer = ((icon_cycle_4.lat>=latitude[0]) & (icon_cycle_4.lat<=latitude[1]))

    icon_cycle_4 = icon_cycle_4.where(indexer,drop=True)
    icon_cycle_4 = icon_cycle_4.load()

    icon_cycle_4 = icon_cycle_4.where(land_sea > 0, np.nan)

    temporal_temp_pear = temporal_auto(icon_cycle_4)
    results_temporal_pear.append(temporal_temp_pear)
    
icon_cycle_4_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")

icon_cycle_4_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/icon_cycle_4_cesm2_grid_30-60N_temporal_corr_nonbinary.nc')


Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...
Processing year 2025...
Processing year 2026...
Processing year 2027...
Processing year 2028...
Processing year 2029...
Processing year 2030...
Processing year 2031...
Processing year 2032...
Processing year 2033...
Processing year 2034...
Processing year 2035...
Processing year 2036...
Processing year 2037...
Processing year 2038...
Processing year 2039...
Processing year 2040...
Processing year 2041...
Processing year 2042...
Processing year 2043...
Processing year 2044...
Processing year 2045...
Processing year 2046...
Processing year 2047...
Processing year 2048...
Processing year 2049...


## IMERG

In [41]:

years = range(2014, 2025)
results_temporal_pear = []
for year in years:
    print(f"Processing year {year}...")

    data_path = Path('/home/jwille/xenon_data/NextGEMS/IMERG/clean/global/cesm2_grid')
    files = sorted([str(f) for f in data_path.glob('daysum_cesm2_interp_IMERG_V07B_final_??_'+str(year)+'.nc4')])[:]
    imerg = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['precipitation']

    imerg = imerg.where(imerg.time.dt.month.isin(months), drop=True)
    indexer = ((imerg.lat>=latitude[0]) & (imerg.lat<=latitude[1]))

    imerg = imerg.where(indexer,drop=True)
    imerg = imerg.load()

    imerg = imerg.where(land_sea > 0, np.nan)

    temporal_temp_pear = temporal_auto(imerg)
    results_temporal_pear.append(temporal_temp_pear)
    
imerg_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")

imerg_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/imerg_cesm2_grid_30-60N_temporal_corr_nonbinary.nc')


Processing year 2014...
Processing year 2015...
Processing year 2016...
Processing year 2017...
Processing year 2018...
Processing year 2019...
Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...


## CESM2

In [ ]:
# Change this to your directory path
model = 'CESM2'
#Source of CESM2 data /net/atmos/data/CESM2-LE/day/PRECT
#directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_past'
directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_future'

# List all file names in the directory (adjust the filter as needed, e.g., for .nc files)
files = [f for f in os.listdir(directory) if f.endswith('.nc')]

# Create a dictionary to group files by simulation number
sim_files = defaultdict(list)


pattern = re.compile(r'LE2-1251\.(\d+)\.cam')

for file in files:
    match = pattern.search(file)
    if match:
        sim_num = match.group(1)
        sim_files[sim_num].append(file)
    else:
        print("Filename did not match expected pattern:", file)
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    # Here you can order or sort the files if the year range matters,
    # e.g., sorting by the start date in the filename (assuming YYYYMMDD is at a fixed position).
    # For example:
    file_list = sorted(file_list)

In [ ]:
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    latitude = [30,60]
    months = [5,6,7,8,9,10]
    results_temporal_pear = []
    results_temporal_spear = []
    
    file_list = sorted(file_list)
    # Create a full file path list
    full_paths = [os.path.join(directory, f) for f in file_list]

    #print(full_paths)
    dataset = xr.open_mfdataset(full_paths, engine='netcdf4',combine='by_coords')['PRECT']
    dataset = dataset * 86400 * 1000
    #dataset = dataset.sel(time=slice('1850', '2015'))  # select time range 
    dataset = dataset.sel(time=slice('2014', '2050'))  # select time range 
    dataset = dataset.where(dataset.time.dt.month.isin(months), drop=True)

    indexer = ((dataset.lat>=latitude[0]) & (dataset.lat<=latitude[1]))
    dataset = dataset.where(indexer,drop=True)
    dataset = dataset.convert_calendar('noleap')
    dataset = dataset.load()

    start_date = pd.Timestamp(''+str(dataset.time.dt.year.values[0]-1)+'-05-01')
    end_date = pd.Timestamp(''+str(dataset.time.dt.year.values[-1])+'-10-31')      
    year_range = dataset.time.dt.year.values[-1] - dataset.time.dt.year.values[0] + 1

    periods = pd.date_range(start_date, end_date, freq='Y')
    num_periods = len(periods)

    time_size = []
#    for i in range(num_periods - 1):
#        # Define the start and end dates for the current period
#        period_start = periods[i]
#        period_end = periods[i + 1]
#        dataset_sel = dataset.sel(time=slice(str(period_start),str(period_end)))
#        if len(dataset_sel.time) > 1:
#            time_size.append(len(dataset_sel.time))
#    print(dataset_sel.time)
#    result = np.zeros([len(time_size),len(dataset.lat),len(dataset.lon)])   


    
    for i in range(num_periods - 1):
        period_start = periods[i]
        period_end = periods[i + 1]
        print(period_start)
        print(period_end)
        dataset_sel = dataset.sel(time=slice(str(period_start),str(period_end)))
        if len(dataset_sel.time) > 1:
            temporal_temp_pear = temporal_auto(dataset_sel,model)

            temporal_temp_pear = temporal_temp_pear.expand_dims(dim={"year": [dataset_sel.time.values[0].year]})

            results_temporal_pear.append(temporal_temp_pear)

    cesm2_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")
    #cesm2_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/cesm2_1850-2014_'+sim+'_30-60N_temporal_corr_nonbinary.nc')


    cesm2_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/cesm2_2014-2049_'+sim+'_30-60N_temporal_corr_nonbinary.nc')
